## Store vector to Postgres

#### *Issues encountered*
- Model(llama3) not support embedding, need use embedding model
- VECTOR_SIZE not match model dimensions, different model output different dimensions, need to match table's dimensions

In [16]:
%pip install langchain langchain-core langchain-ollama langchain-docling

Note: you may need to restart the kernel to use updated packages.


In [17]:
%pip install langchain-postgres sqlalchemy

Note: you may need to restart the kernel to use updated packages.


### Prepare Ollama

Since this example use Ollama embedding, we need prepare Ollama service ahead.

```bash
brew install ollama
brew services start ollama
ollama pull llama3 # use llama3 as embedding model had issue
ollama pull nomic-embed-text #Eebedding model
```

### Load env

In [18]:
import os
from dotenv import load_dotenv

load_dotenv()

print(os.getenv('POSTGRES_PORT'))
print(os.getenv('POSTGRES_USER'))

5432
postgres


### Postgres connection string

In [19]:
CONNECTION_STRING = (
    f"postgresql+asyncpg://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}@{os.getenv('POSTGRES_HOST')}"
    f":{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DB')}"
)

### Create Engine for store, and init table

In [20]:
from langchain_postgres import PGEngine

pg_engine = PGEngine.from_connection_string(url=CONNECTION_STRING)


### Run only once to create vector table

In [21]:
#await pg_engine.ainit_vectorstore_table(
#    table_name=os.getenv('TABLE_NAME'),
#    vector_size=os.getenv('VECTOR_SIZE'),
#)

### Create Embedding for store

In [22]:
from langchain_ollama import OllamaEmbeddings

# use llama3 to embedding cause error
# embedding = OllamaEmbeddings(model="llama3")

# use embed model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text", # dimension=768
    #base_url="http://127.0.0.1:11434",
    #dimensions=1024
)

### Test embedding

In [23]:
import ollama
ollama.embeddings(model='nomic-embed-text', prompt='The sky is blue because of rayleigh scattering')

EmbeddingsResponse(embedding=[0.5892952084541321, 0.40109509229660034, -3.303561210632324, -0.5260301232337952, 0.7490605711936951, 1.518327236175537, -0.12491471320390701, 0.3957957327365875, 0.06742755323648453, -1.1087875366210938, 0.692565381526947, 1.2779479026794434, 1.14599609375, 1.0889348983764648, 0.2507215142250061, 0.2925960123538971, 0.15167468786239624, -0.6339480876922607, -0.21030132472515106, -0.1953614205121994, -1.7962920665740967, -0.6295246481895447, 0.03885599225759506, -0.668717086315155, 1.2614647150039673, 1.2769529819488525, -0.159529909491539, -0.0021719010546803474, -0.29733991622924805, -0.48059698939323425, 1.2050914764404297, -0.6381715536117554, -0.5399741530418396, -1.0355782508850098, 0.6317139863967896, -1.2099220752716064, 0.6831353306770325, -0.05859014391899109, -0.19695024192333221, 0.1275051087141037, -0.014430299401283264, -0.5544432997703552, 0.3514358103275299, 0.04482544586062431, 0.5973907113075256, -0.954983115196228, 0.5075386166572571, 1.

### Create store

In [24]:
from langchain_postgres import PGVectorStore

store = await PGVectorStore.create(
    engine=pg_engine,
    table_name=os.getenv('TABLE_NAME'),
    # schema_name=SCHEMA_NAME,
    embedding_service=embeddings,
    id_column="langchain_id"
)

In [25]:
import uuid

from langchain_core.documents import Document

docs = [
    Document(
        id=str(uuid.uuid4()),
        page_content="Red Apple",
        metadata={"description": "red", "content": "1", "category": "fruit"},
    ),
    Document(
        id=str(uuid.uuid4()),
        page_content="Banana Cavendish",
        metadata={"description": "yellow", "content": "2", "category": "fruit"},
    ),
    Document(
        id=str(uuid.uuid4()),
        page_content="Orange Navel",
        metadata={"description": "orange", "content": "3", "category": "fruit"},
    ),
]

await store.aadd_documents(docs)

['e14a9eaa-816f-4602-a3b1-15648b9509c1',
 '9090543f-e86b-40db-932a-5ee81436627a',
 '8906b398-f208-4593-9904-b0aeca0cd5d4']

In [26]:
import uuid

all_texts = ["Apples and oranges", "Cars and airplanes", "Pineapple", "Train", "Banana"]
metadatas = [{"len": len(t)} for t in all_texts]
ids = [str(uuid.uuid4()) for _ in all_texts]

await store.aadd_texts(all_texts, metadatas=metadatas, ids=ids)

['bcd24626-d25f-4c61-b2fc-3f6aa9d5dbd0',
 '464efc83-07d5-4045-b040-72af170606d9',
 'b4ac4c01-3c3a-46c4-88da-4aaaff136aed',
 'e47a55e8-b0d9-42c3-adbf-947cee900219',
 '846b78bf-5e24-4e8f-a4f5-02729740c052']

### Retrieve by description

In [28]:
query = "I'd like a fruit."
docs = await store.asimilarity_search(query)
for doc in docs:
    print(doc)

page_content='Apples and oranges' metadata={'len': 18}
page_content='Apples and oranges' metadata={'len': 18}
page_content='Banana' metadata={'len': 6}
page_content='Banana' metadata={'len': 6}


### Retrieve by vector

In [33]:
query_vector = embeddings.embed_query(query)
docs = await store.asimilarity_search_by_vector(query_vector, k=2)
for doc in docs:
    print(doc) 

page_content='Apples and oranges' metadata={'len': 18}
page_content='Apples and oranges' metadata={'len': 18}
